In [2]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta


In [3]:
fake = Faker('pt_BR')
random.seed(42)
np.random.seed(42)

TOTAL_CLIENTES = 500
TOTAL_ANALISTAS = 20
TOTAL_SOLICITACOES = 2000

In [ ]:
def gerar_clientes(total):
    clientes = []
    
    for i in range(total):
        score = random.randint(0, 1000)
        
        cliente = {
            'nome': fake.name(),
            'cpf': fake.cpf(),
            'idade': random.randint(18, 75),
            'renda_mensal': round(random.uniform(1500, 30000), 2),
            'score_credito': score,
            'tem_imovel': random.randint(0, 1),
            'tem_veiculo': random.randint(0, 1),
            'tempo_emprego_anos': round(random.uniform(0, 30), 1),
            'qtd_emprestimos_ativos': random.randint(0, 5),
            'historico_inadimplencia': 1 if score < 300 else 0,
            'possui_restricao': 1 if score < 200 else 0,
            'historico_produto': random.choice([
                'emprestimo_pessoal', 
                'credito_consignado', 
                'cartao_credito', 
                None
            ]),
            'data_cadastro': fake.date_time_between(
                start_date='-2y', end_date='now'
            )
        }
        clientes.append(cliente)
    
    return pd.DataFrame(clientes)

df_clientes = gerar_clientes(TOTAL_CLIENTES)
print(f"Clientes gerados: {len(df_clientes)}")
print(df_clientes.head())

Clientes gerados: 500
             nome             cpf  idade  renda_mensal  score_credito  \
0   Sr. Ian Viana  294.701.638-69     25       2212.81            654   
1  Daniel Correia  750.321.689-12     65      26927.12            692   
2   Eduardo Rocha  170.369.285-30     50      18657.53            238   
3   Antony da Luz  130.276.954-52     73       1685.21            828   
4    Yasmin Souza  493.062.815-60     66      11092.94            980   

   tem_imovel  tem_veiculo  tempo_emprego_anos  qtd_emprestimos_ativos  \
0           1            0                 6.7                       5   
1           0            1                 1.0                       0   
2           0            1                 6.6                       4   
3           0            1                10.2                       1   
4           0            1                 2.9                       2   

   historico_inadimplencia  possui_restricao   historico_produto  \
0                        0

In [5]:
def gerar_analistas(total):
    analistas = []
    
    for i in range(total):
        analista = {
            'nome': fake.name(),
            'matricula': f'ANA{str(i+1).zfill(4)}',
            'nivel': random.choice(['junior', 'pleno', 'senior']),
            'turno': random.choice(['manha', 'tarde', 'noite']),
            'ativo': 1
        }
        analistas.append(analista)
    
    return pd.DataFrame(analistas)

df_analistas = gerar_analistas(TOTAL_ANALISTAS)
print(f"Analistas gerados: {len(df_analistas)}")
print(df_analistas)

Analistas gerados: 20
                         nome matricula   nivel  turno  ativo
0               Laura da Mata   ANA0001  junior  noite      1
1                José Camargo   ANA0002  junior  tarde      1
2                Rhavi Pastor   ANA0003   pleno  tarde      1
3                  Isis Sales   ANA0004  junior  noite      1
4                  Otto Porto   ANA0005   pleno  tarde      1
5         Dra. Yasmin Andrade   ANA0006  senior  manha      1
6              Benjamin Silva   ANA0007  junior  noite      1
7                Yago Pacheco   ANA0008  senior  noite      1
8               Luigi da Mata   ANA0009  junior  tarde      1
9              Anthony Campos   ANA0010   pleno  noite      1
10          Alexandre Marques   ANA0011  junior  noite      1
11              Arthur Novaes   ANA0012  senior  tarde      1
12              Milena Guerra   ANA0013  senior  noite      1
13         Srta. Laís Freitas   ANA0014  senior  manha      1
14  Dr. Carlos Eduardo Moraes   ANA0015   pleno 

In [6]:
# Classificar risco baseado em multiplos fatores
def classificar_risco(cliente, valor_solicitado):
    pontuacao = 0
    
    # Score de credito (0 a 40 pontos)
    pontuacao += (cliente['score_credito'] / 1000) * 40
    
    # Renda vs valor solicitado (0 a 20 pontos)
    capacidade = cliente['renda_mensal'] / valor_solicitado
    pontuacao += min(capacidade * 10, 20)
    
    # Historico no banco (0 a 15 pontos)
    if cliente['historico_produto'] is not None:
        pontuacao += 15
    
    # Tempo de emprego (0 a 10 pontos)
    pontuacao += min(cliente['tempo_emprego_anos'], 10)
    
    # Tem imovel (0 a 10 pontos)
    if cliente['tem_imovel'] == 1:
        pontuacao += 10
    
    # Tem veiculo (0 a 5 pontos)
    if cliente['tem_veiculo'] == 1:
        pontuacao += 5
    
    # Penalidades
    if cliente['historico_inadimplencia'] == 1:
        pontuacao -= 30
    
    if cliente['possui_restricao'] == 1:
        pontuacao -= 25
    
    pontuacao -= cliente['qtd_emprestimos_ativos'] * 5
    
    # Garantir que fique entre 0 e 100
    pontuacao = max(0, min(100, pontuacao))
    
    # Classificar
    if pontuacao >= 65:
        return 'baixo', 'automatica'
    elif pontuacao >= 35:
        return 'medio', 'manual'
    else:
        return 'alto', 'recusada'

# Gerar solicitacoes
def gerar_solicitacoes(total, df_clientes, df_analistas):
    solicitacoes = []
    
    analistas_ids = list(range(1, len(df_analistas) + 1))
    
    for i in range(total):
        id_cliente = random.randint(1, len(df_clientes))
        cliente = df_clientes.loc[id_cliente - 1]
        valor_solicitado = round(random.uniform(1000, 50000), 2)
        
        nivel_risco, tipo_analise = classificar_risco(cliente, valor_solicitado)
        
        data_solicitacao = fake.date_time_between(
            start_date='-1y', end_date='now'
        )
        
        if tipo_analise == 'manual':
            id_analista = random.choice(analistas_ids)
        else:
            id_analista = None
            
        solicitacao = {
            'id_cliente': id_cliente,
            'id_analista': id_analista,
            'valor_solicitado': valor_solicitado,
            'prazo_meses': random.choice([12, 24, 36, 48, 60]),
            'finalidade': random.choice([
                'emprestimo_pessoal',
                'financiamento_veiculo',
                'capital_giro',
                'reforma'
            ]),
            'tipo_credito': random.choice([
                'pessoal',
                'consignado'
            ]),
            'nivel_risco': nivel_risco,
            'tipo_analise': tipo_analise,
            'status': 'concluido',
            'data_solicitacao': data_solicitacao,
            'data_inicio_analise': data_solicitacao + timedelta(
                minutes=random.randint(5, 120)
            ),
            'data_conclusao': data_solicitacao + timedelta(
                minutes=random.randint(120, 2880)
            ),
            'canal': random.choice(['app', 'agencia', 'internet_banking'])
        }
        solicitacoes.append(solicitacao)
    
    return pd.DataFrame(solicitacoes)

df_solicitacoes = gerar_solicitacoes(
    TOTAL_SOLICITACOES, df_clientes, df_analistas
)
print(f"Solicitacoes geradas: {len(df_solicitacoes)}")
print(df_solicitacoes.head())

Solicitacoes geradas: 2000
   id_cliente  id_analista  valor_solicitado  prazo_meses  \
0         103          NaN          39119.99           60   
1          83         16.0           2930.88           24   
2         344          NaN           5361.14           12   
3         371          9.0          36455.24           24   
4          52          NaN          40845.74           12   

              finalidade tipo_credito nivel_risco tipo_analise     status  \
0  financiamento_veiculo   consignado        alto     recusada  concluido   
1                reforma   consignado       medio       manual  concluido   
2                reforma      pessoal        alto     recusada  concluido   
3                reforma   consignado       medio       manual  concluido   
4           capital_giro   consignado        alto     recusada  concluido   

     data_solicitacao data_inicio_analise      data_conclusao    canal  
0 2025-06-06 14:02:26 2025-06-06 15:37:26 2025-06-07 20:00:26  agencia